# benchmarking_main.ipynb
**Scalable Benchmarking Framework for Cell Type Annotation Tools (Python Version)**

This is the main execution notebook that orchestrates the benchmarking process:
1. Loads configuration and tool registry
2. Sources helper functions from benchmarking_helpers.ipynb
3. Provides main execution function run_all_tools_cv()
4. Contains example usage patterns

Supports both marker-based and reference-based methods using k-fold cross-validation with comprehensive result aggregation.

This is a Python/scanpy translation of the R Benchmarking.R file.

## Load Required Libraries

In [1]:
# Load required libraries
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
import warnings
from typing import Dict, List, Tuple, Any, Optional, Callable
import os
import pickle
from datetime import datetime

## Load Helper Functions

Note: In a Jupyter environment, you would either:
1. Run the benchmarking_helpers.ipynb notebook first, or
2. Import the functions if converted to a .py module

For now, we'll assume the helper functions are available.

In [2]:
# Load helper functions
%run benchmarking_helpers.ipynb  # Uncomment if running in Jupyter

# Alternative: import from .py version
# from benchmarking_helpers import (
#     calculate_metrics, create_cv_folds, get_fold_data_cached,
#     run_single_tool_cv, aggregate_tool_results, aggregate_all_tools
# )

# Import DL-based tool functions
import sys
sys.path.append('./DL-based')

try:
    from run_scDeepSort import run_scDeepSort_function
    from run_scHash import run_scHash_function
    from run_scDeepinsight import run_scDeepinsight_function
    from run_TOSICA import run_TOSICA_function
    from run_scMMT import run_scMMT_function
    from run_scnym import run_scnym_function
    from run_CIForm import run_CIForm_function
    from run_mtANN import run_mtANN_function
    from run_scTransSort import run_scTransSort_function
    from run_scBalance import run_scBalance_function
    from run_MARS import run_MARS_function
    from run_Cell_BLAST import run_Cell_BLAST_function
    from run_SCANVI import run_SCANVI_function
    from run_ItClust import run_ItClust_function
    from run_ACTINN import run_ACTINN_function
    
    print("✓ All Python DL-based tools imported successfully")
except ImportError as e:
    print(f"⚠ Could not import some Python DL tools: {e}")
    # Set None for any missing tools to prevent errors
    run_scDeepSort_function = None
    run_scHash_function = None
    run_scDeepinsight_function = None
    run_TOSICA_function = None
    run_scMMT_function = None
    run_scnym_function = None
    run_CIForm_function = None
    run_mtANN_function = None
    run_scTransSort_function = None
    run_scBalance_function = None
    run_MARS_function = None
    run_Cell_BLAST_function = None
    run_SCANVI_function = None
    run_ItClust_function = None
    run_ACTINN_function = None

#Not working: scDeepInsight (too complex), MARS (too old), scTransSort(broken), mtANN (broken/too long), ACTINN(outdated tensorflow), ItClust(too old)

Benchmarking helper functions loaded successfully!
✓ All Python DL-based tools imported successfully


## Load run_all_tools_cv

In [3]:
def run_all_tools_cv(adata: ad.AnnData, tools_to_run: List[str] = None, 
                    cv_config: Dict = None) -> Dict[str, Dict]:
    """
    Main Benchmarking Execution Function
    
    Purpose: Orchestrates the complete benchmarking workflow
    Inputs:
      - adata: AnnData object with Ground_Truth_Celltype in obs
      - tools_to_run: List of tool names to benchmark (default: TOOLS_TO_RUN)
      - cv_config: Dict with k, group_by, stratify_by, seed (default: CV_CONFIG)
    Outputs: Dict of results from each tool (pass to aggregate_all_tools())
    Workflow:
      1. Creates k-fold splits once (reused for all tools)
      2. Runs each tool across all folds with error handling
      3. Returns aggregated results ready for comparison
    """
    
    if tools_to_run is None:
        tools_to_run = TOOLS_TO_RUN
    if cv_config is None:
        cv_config = CV_CONFIG
    
    print("=== CELL TYPE ANNOTATION BENCHMARKING ===")
    print(f"Dataset: {len(adata.obs)} cells x {len(adata.var)} genes")
    
    if 'Ground_Truth_Celltype' in adata.obs.columns:
        unique_types = adata.obs['Ground_Truth_Celltype'].unique()
        print(f"Cell types: {', '.join(map(str, unique_types))}")
    
    print(f"Tools to run: {', '.join(tools_to_run)}")
    
    # Create folds once, reuse for all tools
    folds = create_cv_folds(adata, 
                           k=cv_config['k'],
                           group_by=cv_config['group_by'], 
                           stratify_by=cv_config['stratify_by'],
                           seed=cv_config['seed'])
    
    # Initialize results storage
    all_tool_results = {}
    
    # Run each tool
    for tool_name in tools_to_run:
        tool_function = TOOL_REGISTRY.get(tool_name)
        if tool_function is None:
            warnings.warn(f"Tool {tool_name} not found in registry")
            continue
        
        # Run CV for this tool
        tool_results = run_single_tool_cv(adata, tool_function, folds, tool_name)
        all_tool_results[tool_name] = tool_results
    
    return all_tool_results

## Tool Registry and Execution

In [4]:
# Tool registry - populate with converted tool functions
TOOL_REGISTRY = {}

# Add Python DL-based tool functions (first batch)
if run_scDeepSort_function is not None:
    TOOL_REGISTRY["scDeepSort"] = run_scDeepSort_function

if run_scHash_function is not None:
    TOOL_REGISTRY["scHash"] = run_scHash_function

if run_scDeepinsight_function is not None:
    TOOL_REGISTRY["scDeepinsight"] = run_scDeepinsight_function

# Add Python DL-based tool functions (second batch)
if run_TOSICA_function is not None:
    TOOL_REGISTRY["TOSICA"] = run_TOSICA_function

if run_scMMT_function is not None:
    TOOL_REGISTRY["scMMT"] = run_scMMT_function

if run_scnym_function is not None:
    TOOL_REGISTRY["scnym"] = run_scnym_function

if run_CIForm_function is not None:
    TOOL_REGISTRY["CIForm"] = run_CIForm_function

if run_mtANN_function is not None:
    TOOL_REGISTRY["mtANN"] = run_mtANN_function

if run_scTransSort_function is not None:
    TOOL_REGISTRY["scTransSort"] = run_scTransSort_function

if run_scBalance_function is not None:
    TOOL_REGISTRY["scBalance"] = run_scBalance_function

if run_MARS_function is not None:
    TOOL_REGISTRY["MARS"] = run_MARS_function

if run_Cell_BLAST_function is not None:
    TOOL_REGISTRY["Cell_BLAST"] = run_Cell_BLAST_function

if run_SCANVI_function is not None:
    TOOL_REGISTRY["SCANVI"] = run_SCANVI_function

if run_ItClust_function is not None:
    TOOL_REGISTRY["ItClust"] = run_ItClust_function

if run_ACTINN_function is not None:
    TOOL_REGISTRY["ACTINN"] = run_ACTINN_function

# Add other tool functions as they are implemented
# TOOL_REGISTRY["CellTypist"] = run_CellTypist_function

print(f"Tools registered: {list(TOOL_REGISTRY.keys())}")
print(f"Total DL tools available: {len(TOOL_REGISTRY)}")

Tools registered: ['scDeepSort', 'scHash', 'scDeepinsight', 'TOSICA', 'scMMT', 'scnym', 'CIForm', 'mtANN', 'scTransSort', 'scBalance', 'MARS', 'Cell_BLAST', 'SCANVI', 'ItClust', 'ACTINN']
Total DL tools available: 15


## Configuration Section

In [5]:
# Data paths - Datasets (HDF5 files with normalized data)
#ADATA_PATH = "misc/AhernCOVID_Raw/AhernCOVID.h5ad"
ADATA_PATH = "data/realistic_pbmc_challenging_10types_anndata_full.h5ad"
# Cross-validation configuration
CV_CONFIG = {
    'k': 3,                              # Number of folds
    'group_by': None,                    # For grouped CV, set to None for stratified
    'stratify_by': "Ground_Truth_Celltype", # For stratified CV
    'seed': 123                          # Reproducibility
}


## Main Benchmarking Execution Function

In [ ]:
# BASIC WORKFLOW:
#
# 1. Load data
adata = sc.read_h5ad(ADATA_PATH)
#
# 2. Run benchmarking (start with subset for testing)
TOOLS_TO_RUN = [
    'scDeepSort', 'scHash', 'TOSICA', 'scMMT', 'scnym', 'CIForm','scBalance', 'Cell_BLAST', 'SCANVI'
]
results = run_all_tools_cv(adata, tools_to_run= TOOLS_TO_RUN)
#
# 3. Aggregate and compare
final_results = aggregate_all_tools(results)
#
# 4. View results
print(final_results['comparison_table'])
print(final_results['summary_stats'])
#
# 5. Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"benchmarking_results_{timestamp}.pkl", 'wb') as f:
    pickle.dump(final_results, f)
final_results['comparison_table'].to_csv(f"realistic_pbmc_challenging_10types_anndata_full_DL{timestamp}.csv", index=False)


=== CELL TYPE ANNOTATION BENCHMARKING ===
Dataset: 8000 cells x 14909 genes
Cell types: Type_Group5, Type_Group7, Type_Group2, Type_Group1, Type_Group9, Type_Group3, Type_Group4, Type_Group10, Type_Group6, Type_Group8
Tools to run: scDeepSort, scHash, TOSICA, scMMT, scnym, CIForm, scBalance, Cell_BLAST, SCANVI
Mode: Stratified K-Fold CV by 'Ground_Truth_Celltype'.
Fold creation complete.
Fold 1: 2667 cells
Fold 2: 2667 cells
Fold 3: 2666 cells

=== Running scDeepSort ===
  Fold 1/3...
    Computing fold 0 data and markers...
    Training: 5333 cells, Testing: 2667 cells
    Data normalization check: log1p flag found in .uns
    Data already normalized - skipping normalization step


/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: Runt

Found 149090 marker genes across 10 cell types
scDeepSort completed successfully:
  - Total predictions: 2667
  - Unknown predictions: 0
  - Unique predictions: 3
    Completed in 1030.31 seconds, accuracy: 40.38%, peak memory: 6447.18 MB
  Fold 2/3...
    Computing fold 1 data and markers...
    Training: 5333 cells, Testing: 2667 cells
    Data normalization check: log1p flag found in .uns
    Data already normalized - skipping normalization step


/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: Runt

Found 149090 marker genes across 10 cell types
scDeepSort completed successfully:
  - Total predictions: 2667
  - Unknown predictions: 0
  - Unique predictions: 3
    Completed in 1076.42 seconds, accuracy: 38.13%, peak memory: 6447.17 MB
  Fold 3/3...
    Computing fold 2 data and markers...
    Training: 5334 cells, Testing: 2666 cells
    Data normalization check: log1p flag found in .uns
    Data already normalized - skipping normalization step


/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/home/oliver/miniconda3/envs/Benchmarking_Main/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: Runt

Found 149090 marker genes across 10 cell types
